the model stage

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_all = pd.read_csv("../data/processed/train_clean.csv", parse_dates=["date"])
val = pd.read_csv("../data/processed/validation_clean.csv", parse_dates=["date"])
dec = pd.read_csv("../data/processed/december_clean.csv", parse_dates=["date"])

In [3]:
train = train_all[train_all["date"] < "2025-09-01"]
test = train_all[train_all["date"] >= "2025-09-01"]
print(len(train), len(test))

37944 9379


time split: train Jan-Aug (37,944), test Sep-Oct (9,379), same as the real case (predict future months)

In [4]:
median_rpm = train.groupby("equipment")["rate_per_mile"].median()
baseline_pred = test["equipment"].map(median_rpm) * test["distance"]

baseline_mae = (test["posted_rate"] - baseline_pred).abs().mean()
baseline_mape = ((test["posted_rate"] - baseline_pred).abs() / test["posted_rate"]).mean() * 100
print(round(baseline_mae, 2), round(baseline_mape, 2))

175.09 8.09


baseline (median $/mile per equipment x distance): MAE $175, MAPE 8.1%

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor

features = ["distance", "weight", "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
            "equipment_code", "month", "day_of_week"]

model = HistGradientBoostingRegressor(random_state=42)
model.fit(train[features], train["rate_per_mile"])

pred = model.predict(test[features]) * test["distance"]
mae = (test["posted_rate"] - pred).abs().mean()
mape = ((test["posted_rate"] - pred).abs() / test["posted_rate"]).mean() * 100
print(round(mae, 2), round(mape, 2))

50.71 2.15


model without market_index: MAE $51, MAPE 2.2% (~3.5x better than baseline)

In [6]:
features_mi = features + ["market_index"]

model_mi = HistGradientBoostingRegressor(random_state=42)
model_mi.fit(train[features_mi], train["rate_per_mile"])

pred_mi = model_mi.predict(test[features_mi]) * test["distance"]
mae_mi = (test["posted_rate"] - pred_mi).abs().mean()
mape_mi = ((test["posted_rate"] - pred_mi).abs() / test["posted_rate"]).mean() * 100
print(round(mae_mi, 2), round(mape_mi, 2))

60.34 2.52


with market_index: MAE $60, MAPE 2.5% -> worse than without it, so we will drop market_index 

In [7]:
final_model = HistGradientBoostingRegressor(random_state=42)
final_model.fit(train_all[features], train_all["rate_per_mile"])

val["predicted_rate"] = final_model.predict(val[features]) * val["distance"]
dec["predicted_rate"] = final_model.predict(dec[features]) * dec["distance"]

print(val["predicted_rate"].describe())
print(dec[["date", "predicted_rate"]].head(10))

count    12000.000000
mean      2371.548928
std       1365.094768
min        188.951264
25%       1279.541114
50%       2045.989359
75%       3376.145248
max       6855.601387
Name: predicted_rate, dtype: float64
        date  predicted_rate
0 2025-12-01      829.441349
1 2025-12-02      834.967800
2 2025-12-03      838.477964
3 2025-12-04      842.067268
4 2025-12-05      840.586891
5 2025-12-06      833.378028
6 2025-12-07      827.376208
7 2025-12-08      829.441349
8 2025-12-09      834.967800
9 2025-12-10      838.477964


final model trained on all clean data: validation predictions $189-$6,856 (no negatives), December ~$827-$842 with a weekly pattern (mid-week higher)

In [8]:
template = pd.read_csv("../data/validation-predictions-template.csv")
template = template[["load_id"]].merge(val[["load_id", "predicted_rate"]], on="load_id", how="left")
template["predicted_rate"] = template["predicted_rate"].round(2)
template.to_csv("../final_deliverables/validation_predictions.csv", index=False)

dec_out = dec[["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]].copy()
dec_out["predicted_rate"] = dec_out["predicted_rate"].round(2)
dec_out.to_csv("../final_deliverables/december_predictions.csv", index=False)

print(template.shape, template["predicted_rate"].isna().sum(), dec_out.shape)

(12000, 2) 0 (31, 7)


saved validation_predictions.csv (12,000 rows, no missing) and december_predictions.csv (31 rows)

score.py validated both files and created the December chart in final_deliverables/candidate_december.png